# SmartTriage · Componente de Ciencia de Datos

**Predicción del nivel de urgencia en un triage hospitalario**

| | |
|---|---|
| Curso | Ciencia de Datos II — Ing. Naomy Ríos |
| Proyecto base | SmartTriage: app móvil de cola de espera priorizada (Estructura de Datos) |
| Equipo | [Integrantes del equipo] |

Este notebook ejecuta el proceso completo de Ciencia de Datos: obtención de datos,
limpieza / ETL, análisis exploratorio (EDA), entrenamiento y evaluación de modelos
de clasificación, interpretación de resultados y exportación del modelo que la app
consume.

> Se ejecuta de principio a fin en Google Colab sin archivos externos: el dataset
> se genera dentro del propio notebook.

## 1. Problema y objetivos

**Proyecto original.** SmartTriage es una app de sala de emergencias donde el
personal registra pacientes y una **cola de prioridad (heap)** decide el orden de
atención según un nivel de urgencia (1 = Alta, 2 = Media, 3 = Baja) que hoy se
asigna **a mano**.

**Problema.** La asignación manual depende del criterio y la carga de trabajo de
quien recibe. Un error hacia abajo (*sub-triage*: un paciente grave marcado como
Media o Baja) retrasa una atención crítica; un error hacia arriba satura la cola.

**Solución de Ciencia de Datos.** Un modelo de **clasificación** que, a partir de
seis signos clínicos sencillos, **sugiere** el nivel de urgencia y una confianza.
La sugerencia es de apoyo: el personal siempre confirma o cambia la decisión.

**Objetivos.**

1. Construir un dataset representativo del problema (simulado y documentado).
2. Ejecutar ETL, limpieza y EDA sobre ese dataset.
3. Entrenar y comparar modelos de clasificación del nivel de urgencia.
4. Medir el valor agregado frente a la regla lineal simple que ya usa la app.
5. Exportar el modelo e integrarlo al flujo de registro de pacientes.

## 2. Configuración e imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

ETIQUETAS = {1: 'Alta', 2: 'Media', 3: 'Baja'}
print('Entorno listo.')

## 3. Obtención de datos: generación del dataset simulado

No se dispone de expedientes clínicos reales (datos sensibles y protegidos por
ley). Se construye una **población sintética** con dos partes:

* una **regla clínica lineal** (los mismos pesos que la app usa en el cliente), y
* **términos de interacción no lineales** — combinaciones que un triage real sí
  pondera pero una suma de pesos independientes no capta (dolor de pecho *con*
  disnea, fiebre *con* taquicardia ≈ sepsis, adulto mayor frágil, lactante).

Se añade **ruido gaussiano**, así la etiqueta `urgencia` no es una función exacta
de las entradas y el modelo tiene un problema real que aprender (no puede llegar
al 100 %).

> La misma función vive en `data/generar_dataset.py` dentro del repositorio.

**Diccionario de datos**

| Columna | Tipo | Descripción |
|---|---|---|
| `edad` | entero (años) | Edad del paciente |
| `sintoma_principal` | categórica (8) | Motivo principal de consulta |
| `nivel_dolor` | ordinal 1-3 | 1 bajo, 2 medio, 3 alto |
| `dificultad_respiratoria` | booleana | Presencia de disnea |
| `temperatura` | float (°C) | Temperatura corporal |
| `frecuencia_cardiaca` | entero (lpm) | Pulso |
| `urgencia` | objetivo 1-3 | 1 = Alta, 2 = Media, 3 = Baja |

In [ ]:
# Catalogo y pesos. Coinciden con utils/priorityPrediction.ts y con el CHECK de
# la columna sintoma_principal en database/03_patients.sql.
SINTOMAS = [
    'dolor_pecho', 'dificultad_respiratoria', 'fractura', 'dolor_abdominal',
    'mareo', 'fiebre', 'herida', 'otro',
]
PROB_SINTOMA = [0.12, 0.10, 0.13, 0.16, 0.10, 0.15, 0.14, 0.10]
PESO_SINTOMA = {
    'dolor_pecho': 0.95, 'dificultad_respiratoria': 0.90, 'fractura': 0.55,
    'dolor_abdominal': 0.45, 'mareo': 0.40, 'fiebre': 0.35, 'herida': 0.30,
    'otro': 0.25,
}
UMBRAL_ALTA, UMBRAL_MEDIA = 0.62, 0.35


def _clip01(x):
    return np.clip(x, 0.0, 1.0)


def generar_dataset(n=1500, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)

    edad = rng.integers(1, 98, size=n)
    sintoma = rng.choice(SINTOMAS, size=n, p=PROB_SINTOMA)
    nivel_dolor = rng.choice([1, 2, 3], size=n, p=[0.40, 0.35, 0.25])

    p_disnea = np.where(np.isin(sintoma, ['dificultad_respiratoria', 'dolor_pecho']), 0.70, 0.12)
    disnea = rng.random(n) < p_disnea

    temperatura = rng.normal(36.8, 0.5, n)
    temperatura += np.where(sintoma == 'fiebre', rng.normal(1.8, 0.6, n), 0.0)
    temperatura += np.where(disnea, rng.normal(0.3, 0.2, n), 0.0)

    fc = rng.normal(78, 11, n)
    fc += (nivel_dolor - 1) * 7
    fc += np.where(disnea, 14, 0)
    fc += np.where(sintoma == 'dolor_pecho', 10, 0)
    fc = np.clip(fc, 45, 190)

    # Parte lineal: identica a la regla del cliente (baseline).
    score_lineal = (
        ((nivel_dolor - 1) / 2) * 0.30
        + _clip01((temperatura - 36.5) / 4.0) * 0.20
        + _clip01((fc - 70) / 80.0) * 0.15
        + disnea.astype(float) * 0.20
        + np.array([PESO_SINTOMA[s] for s in sintoma]) * 0.10
        + _clip01((edad - 60) / 40.0) * 0.05
    )

    # Parte no lineal: combinaciones que la regla lineal no capta.
    combo_cardiorresp = np.isin(sintoma, ['dolor_pecho', 'dificultad_respiratoria']) & disnea
    posible_sepsis = (temperatura >= 38.0) & (fc >= 110)
    adulto_fragil = (edad >= 75) & (nivel_dolor >= 2)
    lactante = edad <= 2
    herida_leve = (sintoma == 'herida') & (nivel_dolor == 1) & (~disnea)

    score = (
        score_lineal
        + 0.12 * combo_cardiorresp
        + 0.10 * posible_sepsis
        + 0.08 * adulto_fragil
        + 0.06 * lactante
        - 0.05 * herida_leve
        + rng.normal(0.0, 0.05, n)
    )

    urgencia = np.where(score >= UMBRAL_ALTA, 1, np.where(score >= UMBRAL_MEDIA, 2, 3))

    return pd.DataFrame({
        'edad': edad.astype(int),
        'sintoma_principal': sintoma,
        'nivel_dolor': nivel_dolor.astype(int),
        'dificultad_respiratoria': disnea,
        'temperatura': np.round(temperatura, 1),
        'frecuencia_cardiaca': np.round(fc).astype(int),
        'urgencia': urgencia.astype(int),
    })


df = generar_dataset(1500)
df.to_csv('pacientes_simulados.csv', index=False)
print('Dataset:', df.shape)
df.head()

## 4. ETL y limpieza

Los datos crudos casi nunca llegan limpios. Para que el proceso sea realista se
**introducen problemas típicos** en una copia del dataset y luego se corrigen uno
por uno, documentando cada decisión.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
sucio = df.copy()

# 1) Valores faltantes (equipos sin registrar, campos opcionales en blanco).
for col, frac in [('temperatura', 0.04), ('frecuencia_cardiaca', 0.03)]:
    idx = rng.choice(sucio.index, size=int(len(sucio) * frac), replace=False)
    sucio.loc[idx, col] = np.nan

# 2) Valores imposibles: 3.75 = error de tecleo de 37.5; pulso 0.
sucio.loc[rng.choice(sucio.index, 6, replace=False), 'temperatura'] = 3.75
sucio.loc[rng.choice(sucio.index, 4, replace=False), 'frecuencia_cardiaca'] = 0

# 3) Inconsistencia de texto en la categoria.
idx = rng.choice(sucio.index, 20, replace=False)
sucio.loc[idx, 'sintoma_principal'] = '  Dolor_Pecho '

# 4) Filas duplicadas (doble registro del mismo paciente).
sucio = pd.concat([sucio, sucio.sample(15, random_state=RANDOM_STATE)], ignore_index=True)

print('Filas:', len(sucio))
sucio.info()

In [ ]:
# Diagnostico
print('Nulos por columna:')
print(sucio.isna().sum(), '\n')
print('Filas duplicadas:', sucio.duplicated().sum(), '\n')
print('Categorias de sintoma_principal:')
print(sucio['sintoma_principal'].value_counts(dropna=False), '\n')
print('Rango de temperatura:', sucio['temperatura'].min(), '-', sucio['temperatura'].max())
print('Rango de frecuencia_cardiaca:', sucio['frecuencia_cardiaca'].min(), '-', sucio['frecuencia_cardiaca'].max())

In [ ]:
limpio = sucio.copy()

# a) Normalizar la categoria: quitar espacios y pasar a minusculas.
limpio['sintoma_principal'] = (
    limpio['sintoma_principal'].str.strip().str.lower()
)
assert set(limpio['sintoma_principal'].unique()).issubset(set(SINTOMAS))

# b) Marcar como faltantes los valores fisiologicamente imposibles.
limpio.loc[~limpio['temperatura'].between(30, 45), 'temperatura'] = np.nan
limpio.loc[~limpio['frecuencia_cardiaca'].between(30, 220), 'frecuencia_cardiaca'] = np.nan

# c) Imputar con la mediana (robusta a valores extremos).
for col in ['temperatura', 'frecuencia_cardiaca']:
    mediana = limpio[col].median()
    limpio[col] = limpio[col].fillna(mediana)
    print(f'{col}: imputado con mediana = {mediana:.1f}')

# d) Quitar duplicados exactos.
antes = len(limpio)
limpio = limpio.drop_duplicates().reset_index(drop=True)
print(f'Duplicados eliminados: {antes - len(limpio)}')

# e) Tipos definitivos.
limpio['dificultad_respiratoria'] = limpio['dificultad_respiratoria'].astype(int)
limpio['frecuencia_cardiaca'] = limpio['frecuencia_cardiaca'].round().astype(int)

print('\nDataset limpio:', limpio.shape)
print('Nulos restantes:', int(limpio.isna().sum().sum()))
limpio.describe(include='all').T

## 5. Análisis exploratorio (EDA)

In [ ]:
conteo = limpio['urgencia'].value_counts().sort_index()
pct = (conteo / conteo.sum() * 100).round(1)
resumen = pd.DataFrame({'casos': conteo, 'porcentaje': pct})
resumen.index = resumen.index.map(ETIQUETAS)
print(resumen, '\n')

ax = sns.countplot(x='urgencia', data=limpio, order=[1, 2, 3], hue='urgencia',
                   palette='flare', legend=False)
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Alta (1)', 'Media (2)', 'Baja (3)'])
ax.set_title('Distribucion del nivel de urgencia')
ax.set_xlabel('')
plt.show()

In [ ]:
num_cols = ['edad', 'nivel_dolor', 'temperatura', 'frecuencia_cardiaca', 'dificultad_respiratoria']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['edad', 'temperatura', 'frecuencia_cardiaca']):
    for u in [1, 2, 3]:
        sns.kdeplot(limpio.loc[limpio['urgencia'] == u, col], ax=ax, label=ETIQUETAS[u], fill=True, alpha=0.25)
    ax.set_title(f'{col} por nivel de urgencia')
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['temperatura', 'frecuencia_cardiaca', 'edad']):
    sns.boxplot(x='urgencia', y=col, data=limpio, order=[1, 2, 3], ax=ax,
                hue='urgencia', palette='flare', legend=False)
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['Alta', 'Media', 'Baja'])
    ax.set_xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
corr = limpio[num_cols + ['urgencia']].corr(method='spearman')
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', center=0)
plt.title('Correlacion de Spearman (incluye la variable objetivo)')
plt.show()

# urgencia esta codificada 1=Alta..3=Baja, por eso las correlaciones utiles
# con los signos de gravedad salen NEGATIVAS.
print(corr['urgencia'].drop('urgencia').sort_values())

In [ ]:
tabla = pd.crosstab(limpio['sintoma_principal'], limpio['urgencia'].map(ETIQUETAS), normalize='index')
tabla = tabla[['Alta', 'Media', 'Baja']].sort_values('Alta', ascending=False)
print((tabla * 100).round(1))

tabla.plot(kind='barh', stacked=True, figsize=(9, 4), colormap='flare')
plt.title('Composicion de urgencia por sintoma principal')
plt.xlabel('Proporcion')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

**Lectura del EDA.** El dolor de pecho y la dificultad respiratoria concentran la
mayor proporción de urgencias Altas; la herida y el grupo *otro* son mayormente
Bajas. La frecuencia cardíaca y la temperatura suben de forma clara al aumentar la
gravedad, y `nivel_dolor` es la variable numérica con más señal. No hay
colinealidad fuerte entre predictores, así que se pueden usar todos.

## 6. Preparación y modelos

Se comparan dos clasificadores:

* **Árbol de decisión** — interpretable, se puede dibujar y explicar al personal.
* **Random Forest** — conjunto de árboles; suele generalizar mejor y da una
  importancia de variables estable.

La única variable categórica (`sintoma_principal`) se codifica con One-Hot dentro
de un `Pipeline`, para que el mismo objeto haga la transformación y la predicción
cuando se exporte. Se usa `class_weight='balanced'` por el desbalance de clases y
`stratify` en la partición.

In [ ]:
FEATURES = ['edad', 'sintoma_principal', 'nivel_dolor', 'dificultad_respiratoria',
            'temperatura', 'frecuencia_cardiaca']
X = limpio[FEATURES].copy()
y = limpio['urgencia'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y,
)

prep = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['sintoma_principal'])],
    remainder='passthrough',
)

arbol = Pipeline([
    ('prep', prep),
    ('modelo', DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=RANDOM_STATE)),
])
bosque = Pipeline([
    ('prep', prep),
    ('modelo', RandomForestClassifier(n_estimators=300, max_depth=None,
                                      class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
])

arbol.fit(X_train, y_train)
bosque.fit(X_train, y_train)
print('Modelos entrenados.')

In [ ]:
def evaluar(nombre, modelo):
    pred = modelo.predict(X_test)
    return {
        'modelo': nombre,
        'accuracy': accuracy_score(y_test, pred),
        'f1_macro': f1_score(y_test, pred, average='macro'),
    }

metricas = pd.DataFrame([evaluar('Arbol de decision', arbol),
                         evaluar('Random Forest', bosque)]).set_index('modelo')
print(metricas, '\n')

print('Random Forest — reporte de clasificacion:')
print(classification_report(y_test, bosque.predict(X_test),
                            target_names=['Alta (1)', 'Media (2)', 'Baja (3)']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (nombre, modelo) in zip(axes, [('Arbol de decision', arbol), ('Random Forest', bosque)]):
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, modelo.predict(X_test)),
        display_labels=['Alta', 'Media', 'Baja'],
    ).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(nombre)
plt.tight_layout()
plt.show()

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(bosque, X, y, cv=cv, scoring='f1_macro')
print(f'Random Forest — F1 macro (5-fold): {scores.mean():.3f} +/- {scores.std():.3f}')
print('Folds:', np.round(scores, 3))

## 7. Importancia de variables

In [ ]:
nombres = bosque.named_steps['prep'].get_feature_names_out()
importancias = pd.Series(bosque.named_steps['modelo'].feature_importances_, index=nombres)

def base(nombre):
    nombre = nombre.split('__', 1)[-1]
    for s in SINTOMAS:
        if nombre == f'sintoma_principal_{s}':
            return 'sintoma_principal'
    return nombre

imp = importancias.groupby(base).sum().sort_values(ascending=False)
print(imp.round(4), '\n')

imp.sort_values().plot(kind='barh', figsize=(7, 4), color='#0EA5E9')
plt.title('Importancia de variables — Random Forest')
plt.xlabel('Importancia (suma = 1)')
plt.tight_layout()
plt.show()

print('Formato para pegar en utils/priorityPrediction.ts:')
for k, v in imp.items():
    print(f'//   {k:<24} {v:.4f}')

In [ ]:
plt.figure(figsize=(16, 7))
plot_tree(arbol.named_steps['modelo'], max_depth=3, filled=True, fontsize=8,
          feature_names=nombres, class_names=['Alta', 'Media', 'Baja'], impurity=False)
plt.title('Arbol de decision (primeros 3 niveles)')
plt.show()

## 8. Baseline: la regla lineal que ya usa la app

La app calcula hoy una sugerencia con una **suma ponderada de pesos** (sin
interacciones). Es el mismo cálculo que la parte lineal del generador. Se compara
contra el modelo para medir el **valor agregado**, con foco en el error más caro:
el **sub-triage** (urgencia real Alta clasificada como Media o Baja).

In [ ]:
def regla_simple(fila):
    c = lambda x: max(0.0, min(1.0, x))
    score = (
        ((fila['nivel_dolor'] - 1) / 2) * 0.30
        + c((fila['temperatura'] - 36.5) / 4.0) * 0.20
        + c((fila['frecuencia_cardiaca'] - 70) / 80.0) * 0.15
        + (0.20 if fila['dificultad_respiratoria'] else 0.0)
        + PESO_SINTOMA.get(fila['sintoma_principal'], 0.25) * 0.10
        + c((fila['edad'] - 60) / 40.0) * 0.05
    )
    return 1 if score >= UMBRAL_ALTA else (2 if score >= UMBRAL_MEDIA else 3)

pred_regla = X_test.apply(regla_simple, axis=1)
pred_modelo = pd.Series(bosque.predict(X_test), index=X_test.index)

def subtriage(y_true, y_pred):
    grave = y_true == 1
    fallos = ((y_pred[grave] != 1)).sum()
    return fallos, int(grave.sum())

r_ft, r_tot = subtriage(y_test, pred_regla)
m_ft, m_tot = subtriage(y_test, pred_modelo)

comparativa = pd.DataFrame({
    'accuracy': [accuracy_score(y_test, pred_regla), accuracy_score(y_test, pred_modelo)],
    'f1_macro': [f1_score(y_test, pred_regla, average='macro'), f1_score(y_test, pred_modelo, average='macro')],
    'sub_triage (Altas mal clasificadas)': [f'{r_ft}/{r_tot}', f'{m_ft}/{m_tot}'],
}, index=['Regla lineal (app actual)', 'Random Forest (propuesto)'])
comparativa

## 9. Resultados y decisiones

*(Los números exactos salen de la ejecución; lo esperado con esta configuración
es un F1 macro del Random Forest en torno a 0.85-0.92 y una reducción clara del
sub-triage frente a la regla lineal.)*

**Qué decisiones habilita el modelo**

| Decisión | Cómo la apoya el modelo |
|---|---|
| Orden de atención en la cola | La `urgencia` sugerida alimenta directamente el heap de prioridad. |
| Alerta temprana de caso grave | Si el modelo predice Alta con confianza elevada, la app puede resaltar el registro. |
| Revisión de criterio | Cuando la sugerencia y la decisión manual difieren mucho, queda como caso a auditar. |
| Qué datos pedir en el formulario | La importancia de variables justifica los seis campos: si `nivel_dolor`, `frecuencia_cardiaca` y `temperatura` explican la mayor parte, son de captura obligatoria. |

## 10. Valor agregado

* **Mejor toma de decisiones.** El triage deja de depender solo del criterio
  puntual de quien recibe: hay una segunda opinión consistente, entrenada sobre
  toda la población.
* **Menos sub-triage.** El modelo captura combinaciones de riesgo (dolor de pecho
  + disnea, fiebre + taquicardia) que la suma de pesos independiente pasa por
  alto — justo los casos donde un retraso es más peligroso.
* **Eficiencia.** La cola se ordena con una señal más fiable, reduciendo
  reordenamientos manuales.
* **Explicabilidad.** El árbol y la importancia de variables permiten justificar
  cada sugerencia, requisito para que el personal clínico confíe en la herramienta.
* **Diferenciación.** Una cola de espera con priorización asistida por datos es un
  salto frente a una lista de llegada simple.

## 11. Exportación del modelo

Se guarda el `Pipeline` completo (preprocesamiento + modelo) en
`modelo_urgencia.pkl`. La app en producción usa una **destilación** de este modelo
— una regla ligera en `utils/priorityPrediction.ts` — para no depender de un
backend de Python; el `.pkl` queda como referencia y como base para servirlo desde
un endpoint más adelante.

In [ ]:
joblib.dump(bosque, 'modelo_urgencia.pkl')
print('Guardado: modelo_urgencia.pkl')

modelo = joblib.load('modelo_urgencia.pkl')

paciente = pd.DataFrame([{
    'edad': 68,
    'sintoma_principal': 'dolor_pecho',
    'nivel_dolor': 3,
    'dificultad_respiratoria': 1,
    'temperatura': 37.9,
    'frecuencia_cardiaca': 122,
}])

pred = int(modelo.predict(paciente)[0])
proba = modelo.predict_proba(paciente)[0]
print(f'Urgencia sugerida: {pred} ({ETIQUETAS[pred]})')
print('Probabilidades [Alta, Media, Baja]:', np.round(proba, 3))

## 12. Conclusiones, limitaciones y mejoras futuras

**Conclusiones**

* Un modelo de clasificación sencillo (Random Forest sobre seis variables) ordena
  la cola de SmartTriage con más fiabilidad que la regla lineal actual, sobre todo
  reduciendo el sub-triage.
* Las variables de mayor peso (`nivel_dolor`, `frecuencia_cardiaca`,
  `temperatura`, `dificultad_respiratoria`) son baratas de capturar en el
  formulario, lo que hace la solución viable en la práctica.

**Limitaciones**

* El dataset es **simulado**: refleja relaciones plausibles, no una población
  hospitalaria real. Las métricas son orientativas.
* La regla que genera la etiqueta y el baseline comparten la parte lineal, así
  que la ventaja del modelo proviene sobre todo de los términos de interacción
  definidos por el equipo.
* En producción la app usa una destilación del modelo, no el `.pkl` directamente.

**Mejoras futuras**

* Sustituir el dataset simulado por registros reales anonimizados y reentrenar.
* Servir `modelo_urgencia.pkl` desde un endpoint (FastAPI) y que la app lo
  consulte, con la regla local como respaldo sin conexión.
* Guardar en Supabase la urgencia sugerida y la decisión final para medir en
  producción el acuerdo modelo-personal y detectar deriva.
* Probar modelos calibrados (`CalibratedClassifierCV`) para que la confianza
  mostrada al usuario sea una probabilidad real.

**Bibliografía**

* Pedregosa et al. (2011). *Scikit-learn: Machine Learning in Python*. JMLR 12.
* McKinney, W. (2017). *Python for Data Analysis*, 2.ª ed. O'Reilly.
* Gilboy, N. et al. (2020). *Emergency Severity Index (ESI): A Triage Tool for
  Emergency Department Care*. AHRQ.